# Method Assets

Prepares, inspects, and synchronizes the reusable covariance, projector, and layer-statistics assets required by editing methods.

## Environment

In [ ]:
from __future__ import annotations

import gc
import json
import shutil
import sys
from collections.abc import Callable
from pathlib import Path
from time import perf_counter
from typing import Any

import torch
import yaml
from dotenv import load_dotenv
from huggingface_hub import HfApi, hf_hub_download
from tqdm.auto import tqdm

load_dotenv()


def find_project_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for path in (start, *start.parents):
        if (path / "src").exists() and (path / "notebook").exists():
            return path
    raise RuntimeError(f"Could not find project root from {start}")


PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import load_config
from src.editors import load_editor
from src.editors.alpha_edit.alpha_edit_main import get_project
from src.utils import nethook

EVAL_CONFIG_PATH = PROJECT_ROOT / "src" / "evaluate" / "config.yaml"
with open(EVAL_CONFIG_PATH, encoding="utf-8") as file:
    eval_config = yaml.safe_load(file)

model_key = eval_config["model_key"]
REPO_ID = "Savassan/KnowledgeSurgeon"

ALPHA_EDIT_CONFIG = PROJECT_ROOT / "src" / "config" / "alpha_edit" / f"{model_key}.yaml"
alpha_edit_config = load_config(ALPHA_EDIT_CONFIG)

ASSET_DATA_DIR = PROJECT_ROOT / "data" / "assets"
LAYER_STATS_DIR = ASSET_DATA_DIR / "layer_stats"
MODEL_LAYER_STATS_DIR = LAYER_STATS_DIR / model_key
ALPHA_EDIT_PROJECTOR = Path(alpha_edit_config.P_loc)
ASSETS_SETUP_PATH = PROJECT_ROOT / "data" / "evaluation" / model_key / "assets_setup.json"

REMOTE_BASE = "assets"

## Asset Inventory

In [ ]:
def get_file_size(path: Path) -> int:
    return path.stat().st_size if path.exists() else 0


def iter_method_asset_files() -> list[Path]:
    files = []

    if ALPHA_EDIT_PROJECTOR.exists():
        files.append(ALPHA_EDIT_PROJECTOR)

    if MODEL_LAYER_STATS_DIR.exists():
        files.extend(sorted(MODEL_LAYER_STATS_DIR.rglob("*.npz")))

    return sorted(dict.fromkeys(files))

In [ ]:
def summarize_local_assets() -> dict[str, Any]:
    files = iter_method_asset_files()
    return {
        "count": len(files),
        "total_bytes": sum(get_file_size(path) for path in files),
        "projector_exists": ALPHA_EDIT_PROJECTOR.exists(),
        "layer_stats_count": sum(1 for path in files if path.suffix == ".npz"),
        "files": [path.relative_to(PROJECT_ROOT).as_posix() for path in files],
    }

In [ ]:
def list_remote_method_assets(include_layer_stats: bool) -> list[str]:
    api = HfApi()
    prefixes = [f"{REMOTE_BASE}/alpha_edit_projectors/{model_key}/"]
    if include_layer_stats:
        prefixes.append(f"{REMOTE_BASE}/layer_stats/{model_key}/")
    return sorted(
        path
        for path in api.list_repo_files(repo_id=REPO_ID, repo_type="dataset")
        if path.startswith(tuple(prefixes))
    )

## Asset Creation

In [ ]:
def sync_cuda() -> None:
    if torch.cuda.is_available():
        torch.cuda.synchronize()


def clear_cuda_cache() -> None:
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def get_layer_progress(desc: str) -> Callable[..., tqdm]:
    return lambda iterator, total, leave=False: tqdm(iterator, total=total, leave=leave, desc=desc)


def get_projector_shape(model, hparams) -> tuple[int, int, int]:
    weight = nethook.get_parameter(
        model,
        f"{hparams.rewrite_module_tmp.format(hparams.layers[-1])}.weight",
    )
    dim = weight.shape[1]
    return (len(hparams.layers), dim, dim)


def get_relative_path(path: str | Path) -> str:
    resolved = Path(path).resolve()
    try:
        return resolved.relative_to(PROJECT_ROOT).as_posix()
    except ValueError:
        return resolved.as_posix()

In [ ]:
def build_alpha_projector(
    model,
    tok,
    hparams,
    recreate_projector: bool,
    recreate_covariance: bool,
) -> dict[str, Any]:
    p_loc = Path(hparams.P_loc)
    report = {
        "config_path": get_relative_path(ALPHA_EDIT_CONFIG),
        "layers": [],
        "P_loc": get_relative_path(p_loc),
        "mom2_dataset": hparams.mom2_dataset,
        "mom2_n_samples": hparams.mom2_n_samples,
        "mom2_dtype": hparams.mom2_dtype,
        "layer_stats_max_length": hparams.layer_stats_max_length,
        "nullspace_threshold": hparams.nullspace_threshold,
        "existed_before": p_loc.exists(),
    }

    if p_loc.exists() and not recreate_projector:
        report.update({
            "status": "skipped_existing",
            "exists_after": True,
            "file_size_bytes": get_file_size(p_loc),
            "elapsed_seconds": 0.0,
        })
        return report

    start = perf_counter()
    P = torch.zeros(get_projector_shape(model, hparams), device="cpu")
    with tqdm(total=len(hparams.layers), desc="AlphaEdit projector") as projector_pbar:
        for i, layer in enumerate(hparams.layers):
            layer_start = perf_counter()
            layer_name = hparams.rewrite_module_tmp.format(layer)

            def set_project_status(status: str, layer_name: str = layer_name) -> None:
                projector_pbar.set_postfix_str(f"{layer_name}: {status}", refresh=True)

            P[i, :, :] = get_project(
                model,
                tok,
                layer,
                hparams,
                progress=get_layer_progress(f"{layer_name} batches"),
                force_recompute=recreate_covariance,
                status_callback=set_project_status,
            )
            sync_cuda()
            clear_cuda_cache()
            projector_pbar.update(1)
            report["layers"].append({
                "layer": layer,
                "elapsed_seconds": perf_counter() - layer_start,
            })

    p_loc.parent.mkdir(parents=True, exist_ok=True)
    torch.save(P, p_loc)
    del P
    clear_cuda_cache()
    report.update({
        "status": "created" if not report["existed_before"] else "recreated",
        "exists_after": p_loc.exists(),
        "file_size_bytes": get_file_size(p_loc),
        "elapsed_seconds": perf_counter() - start,
    })
    return report

In [ ]:
def create_method_assets(recreate_covariance: bool, recreate_projector: bool) -> dict[str, Any]:
    alpha_hparams = load_config(ALPHA_EDIT_CONFIG)
    editor = load_editor(str(ALPHA_EDIT_CONFIG))
    model = editor.model
    tok = editor.tokenizer
    model.eval()
    sync_cuda()

    asset_start = perf_counter()
    telemetry = {
        "force_recreate": {
            "covariance": recreate_covariance,
            "projector": recreate_projector,
        },
        "assets": {},
    }

    try:
        telemetry["assets"]["alpha_edit_projector"] = build_alpha_projector(
            model,
            tok,
            alpha_hparams,
            recreate_projector=recreate_projector,
            recreate_covariance=recreate_covariance,
        )
    finally:
        del editor, model, tok
        clear_cuda_cache()

    telemetry["total_elapsed_seconds"] = perf_counter() - asset_start
    ASSETS_SETUP_PATH.parent.mkdir(parents=True, exist_ok=True)
    with open(ASSETS_SETUP_PATH, "w", encoding="utf-8") as file:
        json.dump(telemetry, file, indent=2)
    print(f"Saved asset setup: {ASSETS_SETUP_PATH}")
    return telemetry

## Remote Synchronization

In [ ]:
def remote_asset_path(local_file: Path) -> str:
    relative_path = local_file.relative_to(PROJECT_ROOT / "data").as_posix()
    return relative_path

In [ ]:
def upload_method_assets() -> None:
    files_to_upload = iter_method_asset_files()
    if not files_to_upload:
        raise FileNotFoundError(
            "No method assets found. Expected AlphaEdit projector and/or layer_stats .npz files."
        )

    api = HfApi()
    for local_file in files_to_upload:
        relative_path = local_file.relative_to(PROJECT_ROOT).as_posix()
        api.upload_file(
            path_or_fileobj=str(local_file),
            path_in_repo=remote_asset_path(local_file),
            repo_id=REPO_ID,
            repo_type="dataset",
        )
        print(f"Uploaded {relative_path}")

In [ ]:
def local_asset_path(remote_file: str) -> Path:
    if not remote_file.startswith(f"{REMOTE_BASE}/"):
        raise ValueError(f"Unsupported remote asset path: {remote_file}")
    return PROJECT_ROOT / "data" / Path(remote_file)

In [ ]:
def download_method_assets(overwrite: bool, include_layer_stats: bool) -> None:
    remote_files = list_remote_method_assets(include_layer_stats)
    if not remote_files:
        asset_description = "projector and layer statistics" if include_layer_stats else "projector"
        raise FileNotFoundError(
            f"No {asset_description} found for {model_key} in {REPO_ID}"
        )

    for remote_file in remote_files:
        destination = local_asset_path(remote_file)
        relative_path = destination.relative_to(PROJECT_ROOT).as_posix()
        destination.parent.mkdir(parents=True, exist_ok=True)

        if destination.exists() and not overwrite:
            print(f"Skipped {relative_path}")
            continue

        cached_file = hf_hub_download(
            repo_id=REPO_ID,
            repo_type="dataset",
            filename=remote_file,
            force_download=overwrite,
        )
        shutil.copy2(cached_file, destination)
        print(f"Downloaded {relative_path}")

## Manual Actions

In [ ]:
# Include cached layer statistics in remote inventory and download operations.
INCLUDE_LAYER_STATS = False

# Recompute existing AlphaEdit assets instead of reusing them.
RECREATE_COVARIANCE = False
RECREATE_PROJECTOR = False

# Replace local asset files when downloading from Hugging Face.
DOWNLOAD_OVERWRITE = False

In [ ]:
# summarize_local_assets()
# list_remote_method_assets(INCLUDE_LAYER_STATS)

# create_method_assets(RECREATE_COVARIANCE, RECREATE_PROJECTOR)
# upload_method_assets()

# download_method_assets(DOWNLOAD_OVERWRITE, INCLUDE_LAYER_STATS)